# Laboratorul 2 — De la date în Databricks la predicții și dashboard în Power BI

Continuăm Laboratorul 1 (NYC Citywide Payroll + Civil List). Nu repetăm curățarea: folosim direct rezultatul curățat din Lab 1 (`data_lake/clean/`), încărcat într-un Volume Unity Catalog.

**Planul notebook-ului**

1. **Date în cloud și Delta Lake** — subset de ~90 MB → tabele Delta `bronze_*` și `silver_payroll_features`.
2. **Model predictiv (Spark MLlib)** — clasificare: *va avea angajatul ore suplimentare în anul fiscal?* Un singur model (regresie logistică), evaluat pe un an fiscal separat, salvat și reîncărcat; predicții în Delta.
3. **Fiabilitate și monitorizare** — partiționare, copie de siguranță (DEEP CLONE), întrerupere simulată a fluxului de predicții și reluare fără pierderi/duplicate, jurnal de monitorizare.
4. **Date pentru dashboard** — tabele agregate pentru Power BI și demonstrarea actualizării.

Rulare: Databricks Free Edition (serverless). Notebook-ul nu conține parole sau tokenuri.

## 1. Date în cloud și Delta Lake

**Selecția datelor.** Din Lab 1 păstrăm cele două ieșiri curățate: `payroll` (≈80 MB, 6,8 mil. rânduri) și `civil_list` (≈10 MB, 3,2 mil. rânduri), în total ≈90 MB — aproape de ținta de 100 MB. Sunt deja convertite (salarii numerice, ani întregi, null-uri tratate), deci nu refacem prelucrarea.

**Organizarea (arhitectură medallion).**
- `bronze_payroll`, `bronze_civil_list` — copia fidelă a fișierelor curățate, plus `record_id` (cheie stabilă) și `ingested_at`.
- `silver_payroll_features` — rândurile utilizabile pentru model, cu eticheta și variabilele derivate, partiționate după `fiscal_year`.
- `gold_*` — predicții și agregări pentru dashboard (secțiunile 2–4).

Fișierele sursă stau în Volume-ul `workspace.lab2.raw`; modelul și checkpoint-urile în `workspace.lab2.artifacts`.

In [ ]:
import json
import time
from datetime import datetime, timezone

from pyspark.sql import functions as F

CATALOG, SCHEMA = "workspace", "lab2"
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
ART = f"/Volumes/{CATALOG}/{SCHEMA}/artifacts"
MODEL_PATH = f"{ART}/models/ot_classifier"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")


def t(name):
    return f"{CATALOG}.{SCHEMA}.{name}"


source_files = [
    f for folder in ("payroll", "civil_list")
    for f in dbutils.fs.ls(f"{RAW}/{folder}") if f.name.endswith(".parquet")
]
total_mb = sum(f.size for f in source_files) / 1024 ** 2
print(f"Fișiere Parquet sursă: {len(source_files)} | dimensiune totală: {total_mb:.1f} MB")

In [ ]:
payroll_raw = spark.read.parquet(f"{RAW}/payroll")
civil_raw = spark.read.parquet(f"{RAW}/civil_list")

(payroll_raw
    .withColumn("record_id", F.monotonically_increasing_id())
    .withColumn("ingested_at", F.current_timestamp())
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(t("bronze_payroll")))

(civil_raw
    .withColumn("ingested_at", F.current_timestamp())
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(t("bronze_civil_list")))

for name in ("bronze_payroll", "bronze_civil_list"):
    print(f"{name}: {spark.table(t(name)).count():,} rânduri")
spark.table(t("bronze_payroll")).printSchema()

### Tabelul `silver_payroll_features`

**Eticheta:** `label = 1` dacă înregistrarea are `ot_hours > 0` (angajatul a făcut ore suplimentare în anul fiscal).

**Variabile folosite** — doar informații cunoscute la începutul anului, pentru a evita *leakage*: agenția, borough-ul, tipul de plată, funcția, salariul de bază și anul. Nu folosim `ot_hours`, `total_ot_paid`, `total_other_pay` sau `regular_gross_paid`, care sunt rezultate ale anului respectiv.

Funcțiile (`title_description`) au mii de valori; păstrăm primele 300 ca frecvență și grupăm restul în `OTHER`. Salariul de bază are semnificații diferite după `pay_basis` (anual vs. orar/zilnic), așa că îl împărțim în două variabile logaritmice.

In [ ]:
bronze = spark.table(t("bronze_payroll"))

top_titles = (bronze.groupBy("title_description").count()
              .orderBy(F.desc("count")).limit(300)
              .select("title_description", F.lit(True).alias("is_top_title")))

is_annual = F.lower(F.trim(F.col("pay_basis"))) == "per annum"
silver = (bronze
    .filter(F.col("fiscal_year").isNotNull() & F.col("ot_hours").isNotNull()
            & F.col("base_salary").isNotNull() & (F.col("base_salary") > 0))
    .join(F.broadcast(top_titles), "title_description", "left")
    .withColumn("agency_name", F.coalesce(F.upper(F.trim("agency_name")), F.lit("UNKNOWN")))
    .withColumn("work_location_borough",
                F.coalesce(F.upper(F.trim("work_location_borough")), F.lit("UNKNOWN")))
    .withColumn("pay_basis", F.coalesce(F.trim("pay_basis"), F.lit("UNKNOWN")))
    .withColumn("title_group", F.when(F.col("is_top_title"), F.col("title_description"))
                                .otherwise(F.lit("OTHER")))
    .withColumn("log_salary_annual", F.when(is_annual, F.log1p("base_salary")).otherwise(0.0))
    .withColumn("log_salary_rate", F.when(~is_annual, F.log1p("base_salary")).otherwise(0.0))
    .withColumn("label", (F.col("ot_hours") > 0).cast("double"))
    .select("record_id", "fiscal_year", "agency_name", "work_location_borough", "pay_basis",
            "title_group", "base_salary", "log_salary_annual", "log_salary_rate",
            "ot_hours", "total_ot_paid", "label"))

(silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .partitionBy("fiscal_year").saveAsTable(t("silver_payroll_features")))

silver = spark.table(t("silver_payroll_features"))
print(f"silver_payroll_features: {silver.count():,} rânduri")
display(silver.groupBy("fiscal_year")
        .agg(F.count("*").alias("records"), F.round(F.avg("label"), 3).alias("ot_rate"))
        .orderBy("fiscal_year"))

## 2. Un model predictiv — clasificarea orelor suplimentare

**Problema.** Pentru fiecare înregistrare salarială prezicem probabilitatea ca angajatul să facă ore suplimentare în anul fiscal. Agregată pe agenții, probabilitatea dă numărul *așteptat* de angajați cu ore suplimentare — util pentru planificarea bugetului de overtime și a personalului.

**Evaluare pe date separate.** Facem o separare temporală: antrenăm pe anii fiscali anteriori și testăm pe ultimul an disponibil. E mai realist decât o separare aleatoare, pentru că modelul va fi folosit tocmai pentru un an pe care nu l-a văzut.

**Modelul.** Un singur model: `LogisticRegression` într-un `Pipeline` MLlib (StringIndexer → OneHotEncoder → VectorAssembler → LogisticRegression). Regresia logistică tratează bine variabilele categoriale cu multe valori (one-hot) și coeficienții ei sunt ușor de interpretat.

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array

years = sorted(r["fiscal_year"] for r in silver.select("fiscal_year").distinct().collect())
TEST_YEAR = years[-1]
TRAIN_FRACTION = 0.35  # eșantion din anii de antrenare, pentru timpi rezonabili pe serverless

train_df = silver.filter(F.col("fiscal_year") < TEST_YEAR).sample(False, TRAIN_FRACTION, seed=42)
test_df = silver.filter(F.col("fiscal_year") == TEST_YEAR)
print(f"Ani disponibili: {years[0]}–{years[-1]} | an de test: {TEST_YEAR}")
print(f"Antrenare: {train_df.count():,} rânduri | Test: {test_df.count():,} rânduri")

categorical = ["agency_name", "work_location_borough", "pay_basis", "title_group"]
numeric = ["log_salary_annual", "log_salary_rate"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
            for c in categorical]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical],
                        outputCols=[f"{c}_ohe" for c in categorical], handleInvalid="keep")
assembler = VectorAssembler(inputCols=[f"{c}_ohe" for c in categorical] + numeric,
                            outputCol="features")
classifier = LogisticRegression(featuresCol="features", labelCol="label",
                                maxIter=40, regParam=0.001)

started = time.time()
model = Pipeline(stages=indexers + [encoder, assembler, classifier]).fit(train_df)
print(f"Model antrenat în {time.time() - started:.0f} s")

### Salvarea și reîncărcarea modelului

Salvăm pipeline-ul complet (inclusiv indexarea categoriilor) în Volume și îl reîncărcăm. Toate predicțiile de mai jos și fluxul din secțiunea 3 folosesc **modelul reîncărcat**, ceea ce dovedește că este reutilizabil fără reantrenare.

In [ ]:
model.write().overwrite().save(MODEL_PATH)
loaded_model = PipelineModel.load(MODEL_PATH)
MODEL_VERSION = f"ot_lr_{datetime.now(timezone.utc):%Y%m%d%H%M}"
print(f"Model salvat și reîncărcat din: {MODEL_PATH} (versiune {MODEL_VERSION})")


def add_prediction_columns(scored, model_version):
    return (scored
        .withColumn("ot_probability", F.round(vector_to_array("probability")[1], 4))
        .withColumn("prediction", F.col("prediction").cast("int"))
        .withColumn("label", F.col("label").cast("int"))
        .withColumn("model_version", F.lit(model_version))
        .withColumn("scored_at", F.current_timestamp())
        .select("record_id", "fiscal_year", "agency_name", "work_location_borough",
                "pay_basis", "title_group", "base_salary", "ot_hours", "total_ot_paid",
                "label", "prediction", "ot_probability", "model_version", "scored_at"))


test_scored = add_prediction_columns(loaded_model.transform(test_df), MODEL_VERSION)

### Evaluare pe anul de test: 2 metrici

- **ROC AUC** — cât de bine ordonează modelul angajații după risc, independent de prag (0,5 = aleator, 1 = perfect).
- **F1 (clasa pozitivă)** — echilibrul dintre precizie și recall la pragul 0,5.

Comparăm și cu un reper naiv: rata reală de overtime din anul de test (acuratețea dacă am prezice mereu clasa majoritară).

In [ ]:
auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="ot_probability",
                                    metricName="areaUnderROC").evaluate(test_scored)

cm = (test_scored.groupBy("label", "prediction").count().toPandas()
      .set_index(["label", "prediction"])["count"])
tp, fp = cm.get((1, 1), 0), cm.get((0, 1), 0)
fn, tn = cm.get((1, 0), 0), cm.get((0, 0), 0)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
accuracy = (tp + tn) / (tp + tn + fp + fn)
positive_rate = (tp + fn) / (tp + tn + fp + fn)
baseline_accuracy = max(positive_rate, 1 - positive_rate)

print(f"ROC AUC:   {auc:.3f}")
print(f"F1:        {f1:.3f}  (precizie {precision:.3f}, recall {recall:.3f})")
print(f"Acuratețe: {accuracy:.3f}  vs. reper clasa majoritară {baseline_accuracy:.3f}")
print(f"Rata reală de overtime în {TEST_YEAR}: {positive_rate:.3f}")

metrics_row = [(MODEL_VERSION, int(TEST_YEAR), float(auc), float(f1), float(precision),
                float(recall), float(accuracy), float(baseline_accuracy))]
(spark.createDataFrame(metrics_row, "model_version string, test_year int, roc_auc double, "
                       "f1 double, precision double, recall double, accuracy double, "
                       "baseline_accuracy double")
    .withColumn("evaluated_at", F.current_timestamp())
    .write.format("delta").mode("append").saveAsTable(t("gold_model_metrics")))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

agency_check = (test_scored.groupBy("agency_name")
    .agg(F.count("*").alias("records"), F.avg("label").alias("actual_rate"),
         F.avg("ot_probability").alias("predicted_rate"))
    .orderBy(F.desc("records")).limit(15).toPandas())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 2]})

matrix = np.array([[tn, fp], [fn, tp]])
ax1.imshow(matrix, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax1.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center",
                 color="white" if matrix[i, j] > matrix.max() / 2 else "black")
ax1.set_xticks([0, 1], ["Prezis: fără OT", "Prezis: OT"])
ax1.set_yticks([0, 1], ["Real: fără OT", "Real: OT"])
ax1.set_title(f"Matricea de confuzie — anul de test {TEST_YEAR}")

y = np.arange(len(agency_check))
ax2.barh(y - 0.2, agency_check["actual_rate"], height=0.4, label="Rata reală", color="#4C72B0")
ax2.barh(y + 0.2, agency_check["predicted_rate"], height=0.4, label="Rata prezisă",
         color="#DD8452")
ax2.set_yticks(y, [name[:35] for name in agency_check["agency_name"]])
ax2.invert_yaxis()
ax2.set_xlabel("Proporția angajaților cu ore suplimentare")
ax2.set_title("Rata reală vs. prezisă — top 15 agenții după volum")
ax2.legend()
plt.tight_layout()
plt.show()

### Interpretarea performanței

- **AUC** arată cât de bine separă modelul angajații cu ore suplimentare de cei fără. Valorile peste ~0,8 înseamnă că funcția, agenția și salariul explică o mare parte din probabilitatea de overtime.
- **F1** arată calitatea deciziilor la pragul 0,5. Precizia și recall-ul indică tipul de eroare dominant: fals pozitivi (overtime estimat în plus) sau fals negativi (overtime ratat).
- **Graficul pe agenții** verifică dacă probabilitățile medii sunt apropiate de ratele reale, adică dacă modelul este bine *calibrat* la nivelul folosit în dashboard.

**Utilitatea predicțiilor.** Suma probabilităților pe agenție/borough estimează câți angajați vor face ore suplimentare în anul următor. Asta ajută la alocarea bugetului de overtime și semnalează agențiile unde ar fi mai ieftin să angajezi decât să plătești ore suplimentare. Modelul **nu** estimează suma plătită și nu explică *de ce* apar orele suplimentare.

### Predicții salvate în Delta pentru dashboard

Aplicăm modelul reîncărcat pe toți anii și salvăm rezultatul în `gold_predictions`, partiționat după `fiscal_year`. Coloana `split` indică dacă rândul provine din perioada de antrenare (`train`) sau din anul de test (`test`). Doar rândurile `test` reprezintă performanța reală pe date nevăzute.

In [ ]:
gold_predictions = (add_prediction_columns(loaded_model.transform(silver), MODEL_VERSION)
    .withColumn("split", F.when(F.col("fiscal_year") == TEST_YEAR, "test").otherwise("train")))

(gold_predictions.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .partitionBy("fiscal_year").saveAsTable(t("gold_predictions")))

print(f"gold_predictions: {spark.table(t('gold_predictions')).count():,} rânduri")
display(spark.table(t("gold_predictions")).filter(F.col("split") == "test").limit(10))

## 3. Organizare, fiabilitate și monitorizare

### 3.1 Partiționare

`gold_predictions` (și `silver_payroll_features`) sunt partiționate după `fiscal_year`: fiecare an are directorul lui de fișiere. Interogările filtrate pe an (cum face dashboard-ul) citesc doar partițiile necesare (*partition pruning*), iar un an poate fi rescris sau restaurat fără să fie atinși ceilalți.

In [ ]:
detail = spark.sql(f"DESCRIBE DETAIL {t('gold_predictions')}").first()
print("Coloane de partiționare:", detail["partitionColumns"])
print(f"Fișiere: {detail['numFiles']} | dimensiune: {detail['sizeInBytes'] / 1024 ** 2:.1f} MB")

display(spark.table(t("gold_predictions")).groupBy("fiscal_year")
        .agg(F.count("*").alias("records")).orderBy("fiscal_year"))

print("Plan de execuție pentru un filtru pe an (căutați PartitionFilters):")
spark.table(t("gold_predictions")).filter(F.col("fiscal_year") == TEST_YEAR).explain()

### 3.2 Copie de siguranță separată

Creăm `gold_predictions_backup` cu **DEEP CLONE**: o copie fizică independentă (date + metadate). Spre deosebire de *time travel*, copia rămâne validă și dacă tabelul sursă este șters sau curățat cu `VACUUM`. Demonstrăm apoi o ștergere accidentală și recuperarea rândurilor din copie.

In [ ]:
spark.sql(f"CREATE OR REPLACE TABLE {t('gold_predictions_backup')} DEEP CLONE {t('gold_predictions')}")

source_count = spark.table(t("gold_predictions")).count()
backup_count = spark.table(t("gold_predictions_backup")).count()
print(f"Sursă: {source_count:,} | Backup: {backup_count:,} | identice: {source_count == backup_count}")

# Incident simulat: se șterg din greșeală predicțiile anului de test.
spark.sql(f"DELETE FROM {t('gold_predictions')} WHERE fiscal_year = {TEST_YEAR}")
print(f"După ștergerea accidentală: {spark.table(t('gold_predictions')).count():,} rânduri")

# Recuperare idempotentă din backup: se inserează doar rândurile lipsă.
spark.sql(f'''
    MERGE INTO {t('gold_predictions')} AS target
    USING (SELECT * FROM {t('gold_predictions_backup')} WHERE fiscal_year = {TEST_YEAR}) AS source
    ON target.record_id = source.record_id AND target.fiscal_year = source.fiscal_year
    WHEN NOT MATCHED THEN INSERT *
''')
restored_count = spark.table(t("gold_predictions")).count()
print(f"După recuperare: {restored_count:,} rânduri | complet: {restored_count == source_count}")

**Partiționare și backup vs. sharding și replicare automată**

| Concept | Ce face | Unde apare aici |
|---|---|---|
| **Partiționare** | Împarte *logic* un tabel după o coloană (an) în grupuri de fișiere, în același sistem de stocare. | `partitionBy("fiscal_year")` |
| **Sharding** | Împarte datele pe *noduri/servere diferite*, fiecare deținând o parte. Scalează scrierea și stocarea pe orizontală. | Nu se aplică direct; Spark distribuie calculul, iar stocarea obiect (S3) distribuie fișierele intern. |
| **Copie de siguranță** | Copie separată, făcută la un moment ales, pentru recuperare după erori umane sau corupere. | `DEEP CLONE` → `gold_predictions_backup` |
| **Replicare automată** | Sistemul păstrează continuu mai multe copii identice pe servere/zone diferite pentru disponibilitate. | Asigurată transparent de stocarea cloud din spatele Databricks; nu o administrăm noi. |

Legătura: partiționarea și sharding-ul răspund la *cum împărțim* datele, iar backup-ul și replicarea la *câte copii păstrăm*. Replicarea copiază instantaneu și o ștergere greșită (o propagă pe toate copiile), de aceea un backup la un moment dat rămâne necesar.

### 3.3 Toleranță la erori: întrerupere simulată și reluare

Fluxul de predicții primește înregistrări noi (simulate din anul de test), în 4 loturi. **Spark Structured Streaming** citește tabelul `stream_incoming`, aplică modelul reîncărcat în `foreachBatch` și scrie în `gold_predictions_stream`.

Fluxul rulează în notebook-ul separat `lab2_stream_worker`, pornit ca job cu `dbutils.notebook.run`, așa cum ar rula un job de producție programat. Astfel, căderea simulată oprește doar acel job, iar acest notebook verifică rezultatul și pornește reluarea.

Garanții:
- **Checkpoint-ul** memorează ce loturi au fost planificate (`offsets/`) și finalizate (`commits/`). La repornire, un lot început dar nefinalizat este procesat din nou → **fără pierderi**.
- Scrierea se face cu **MERGE pe `record_id`** (idempotent). Dacă un lot reprocesat era deja scris, nu se inserează a doua oară → **fără duplicate**.

Simulăm cel mai dificil caz: job-ul cade **după** ce lotul 1 a fost scris în tabel, dar **înainte** ca checkpoint-ul să-l marcheze ca finalizat.

In [ ]:
STREAM_SOURCE = t("stream_incoming")
STREAM_SINK = t("gold_predictions_stream")
MONITOR_LOG = t("pipeline_monitoring_log")
CHECKPOINT = f"{ART}/checkpoints/predictions_stream"
WORKER = "./lab2_stream_worker"

# Pornim de la zero, ca demonstrația să fie reproductibilă.
for table in (STREAM_SOURCE, STREAM_SINK, MONITOR_LOG):
    spark.sql(f"DROP TABLE IF EXISTS {table}")
dbutils.fs.rm(CHECKPOINT, True)

incoming = (spark.table(t("silver_payroll_features"))
            .filter(F.col("fiscal_year") == TEST_YEAR)
            .sample(False, 0.25, seed=7))
chunks = incoming.randomSplit([1.0, 1.0, 1.0, 1.0], seed=11)

incoming.limit(0).write.format("delta").saveAsTable(STREAM_SOURCE)
for chunk in chunks:  # fiecare append = un commit cu un singur fișier = un lot de streaming
    chunk.repartition(1).write.format("delta").mode("append").saveAsTable(STREAM_SOURCE)

(spark.table(t("gold_predictions")).drop("split").limit(0)
    .write.format("delta").partitionBy("fiscal_year").saveAsTable(STREAM_SINK))
spark.sql(f'''CREATE TABLE {MONITOR_LOG} (run_label STRING, batch_id BIGINT, status STRING,
              input_rows BIGINT, inserted_rows BIGINT, duration_s DOUBLE, error STRING,
              logged_at TIMESTAMP)''')

source_total = spark.table(STREAM_SOURCE).count()
print(f"Înregistrări noi de procesat: {source_total:,} în {len(chunks)} loturi")

**Rularea 1 — job-ul cade în timpul lotului 1.**

In [ ]:
try:
    dbutils.notebook.run(WORKER, 1800, {"run_label": "run1", "crash_after_batch": "1"})
    print("Neașteptat: rularea 1 s-a terminat fără eroare.")
except Exception as exc:
    print("Rularea 1 s-a oprit cu eroare (așteptat):", str(exc).splitlines()[0][:250])

committed = sorted(int(f.name) for f in dbutils.fs.ls(f"{CHECKPOINT}/commits") if f.name.isdigit())
planned = sorted(int(f.name) for f in dbutils.fs.ls(f"{CHECKPOINT}/offsets") if f.name.isdigit())
print(f"Loturi planificate (offsets): {planned}")
print(f"Loturi finalizate (commits):  {committed}")
print(f"Rânduri în sink după cădere: {spark.table(STREAM_SINK).count():,} din {source_total:,}")
display(spark.table(MONITOR_LOG).orderBy("logged_at"))

**Rularea 2 — reluare cu același checkpoint.** Spark reia lotul 1 (planificat, dar nefinalizat), apoi procesează loturile rămase. MERGE-ul nu reintroduce rândurile lotului 1 deja scrise.

In [ ]:
run2_result = json.loads(
    dbutils.notebook.run(WORKER, 1800, {"run_label": "run2", "crash_after_batch": ""}))

sink = spark.table(STREAM_SINK)
sink_total = sink.count()
duplicates = sink.groupBy("record_id").count().filter("count > 1").count()
missing = spark.table(STREAM_SOURCE).join(sink, "record_id", "left_anti").count()

print(f"Sursă: {source_total:,} | Sink: {sink_total:,}")
print(f"Rânduri lipsă: {missing} | record_id duplicate: {duplicates}")
assert sink_total == source_total and missing == 0 and duplicates == 0
print("✔ Reluare fără pierderi și fără duplicate.")
display(spark.table(MONITOR_LOG).orderBy("logged_at"))

Jurnalul arată lotul 1 de două ori: în `run1` cu `FAILED` (rândurile au fost scrise înainte de cădere), iar în `run2` cu `OK` și **0 rânduri inserate** — reprocesarea nu a produs duplicate. Fără MERGE (un simplu `append`), lotul 1 ar fi apărut de două ori în tabel.

### 3.4 Monitorizare

Urmărim execuțiile pe trei niveluri:

1. **Progresul streaming** (`query.recentProgress`, returnat de job-ul worker): rânduri la intrare, durata fiecărui lot, rânduri/secundă. Un lot neobișnuit de lent indică o problemă de performanță.
2. **Jurnalul propriu** `pipeline_monitoring_log`: un rând pentru fiecare lot, cu status, erori și durată. Poate fi interogat și folosit pentru alerte (de exemplu: orice status diferit de `OK` sau o durată peste prag).
3. **Istoricul Delta** (`DESCRIBE HISTORY`): fiecare scriere în tabel, cu operația și metricile ei (`numTargetRowsInserted`). Arată exact ce a scris fiecare lot.

În interfața Databricks se folosesc în plus: **Jobs & Pipelines → Runs** (cele două rulări ale `lab2_stream_worker` apar acolo, prima cu status *Failed* și eroarea completă) și **Query profile / Query History** (pașii lenți, volumul citit, spill pe disc).

In [ ]:
import pandas as pd

progress_pd = pd.DataFrame(run2_result["progress"])
print("Progresul fluxului la reluare (query.recentProgress):")
display(progress_pd)

log = spark.table(MONITOR_LOG)
avg_duration = log.filter("status = 'OK'").agg(F.avg("duration_s")).first()[0]
alerts = log.filter((F.col("status") != "OK") | (F.col("duration_s") > 2 * avg_duration))
print(f"Durată medie per lot reușit: {avg_duration:.1f} s. Loturi care necesită atenție:")
display(alerts.select("run_label", "batch_id", "status", "duration_s", "error"))

display(spark.sql(f"DESCRIBE HISTORY {STREAM_SINK}")
        .select("version", "timestamp", "operation",
                F.col("operationMetrics.numTargetRowsInserted").alias("rows_inserted"),
                F.col("operationMetrics.executionTimeMs").alias("execution_ms"))
        .orderBy("version"))

## 4. Date pentru dashboard-ul Power BI

Power BI importă tabele mici și agregate, nu cele 6,8 milioane de rânduri. Pregătim:

- `gold_dashboard_agency_year` — pe an × agenție × borough × tip de plată: număr de înregistrări, overtime real, overtime prezis (suma probabilităților), rate, plăți OT și salariul mediu.
- `gold_model_metrics` — metricile modelului (deja scrise în secțiunea 2).

Coloana `refreshed_at` arată în raport când au fost actualizate datele.

In [ ]:
def build_dashboard_table():
    (spark.table(t("gold_predictions"))
        .groupBy("fiscal_year", "split", "agency_name", "work_location_borough", "pay_basis")
        .agg(F.count("*").alias("records"),
             F.sum("label").alias("actual_ot_employees"),
             F.round(F.sum("ot_probability"), 1).alias("expected_ot_employees"),
             F.sum("prediction").alias("predicted_ot_employees"),
             F.round(F.avg("label"), 4).alias("actual_ot_rate"),
             F.round(F.avg("ot_probability"), 4).alias("predicted_ot_rate"),
             F.round(F.sum("total_ot_paid"), 2).alias("total_ot_paid"),
             F.round(F.sum("ot_hours"), 1).alias("total_ot_hours"),
             F.round(F.avg("base_salary"), 2).alias("avg_base_salary"))
        .withColumn("refreshed_at", F.current_timestamp())
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(t("gold_dashboard_agency_year")))
    return spark.table(t("gold_dashboard_agency_year"))


dashboard = build_dashboard_table()
print(f"gold_dashboard_agency_year: {dashboard.count():,} rânduri")
display(dashboard.filter(F.col("split") == "test")
        .groupBy("agency_name")
        .agg(F.sum("records").alias("records"), F.sum("actual_ot_employees").alias("actual"),
             F.sum("expected_ot_employees").alias("expected"))
        .orderBy(F.desc("records")).limit(10))

### Demonstrarea actualizării

1. În Power BI, raportul afișează `refreshed_at` și totalurile curente.
2. Rulați din nou celula de mai sus (sau întregul notebook): tabelul se rescrie cu un nou `refreshed_at`.
3. În Power BI apăsați **Home → Refresh**: cardul „Ultima actualizare” și graficele se actualizează.

### Conectarea Power BI

- **Server hostname** și **HTTP Path**: *SQL Warehouses → Serverless Starter Warehouse → Connection details*.
- **Power BI Desktop → Get Data → Azure Databricks** (funcționează și pentru Databricks pe AWS), mod **Import**.
- Autentificare cu un Personal Access Token, introdus doar în Power BI (nu în notebook).
- Tabele: `workspace.lab2.gold_dashboard_agency_year` și `workspace.lab2.gold_model_metrics`.

### Decizii sprijinite de dashboard

1. **Alocarea bugetului de overtime pe agenții** — numărul așteptat de angajați cu ore suplimentare (din predicții) × plata medie OT pe angajat arată unde trebuie rezervat buget pentru anul următor.
2. **Unde merită angajări în loc de ore suplimentare** — agențiile/borough-urile cu rată prezisă mare și plăți OT mari sunt candidate pentru posturi noi sau redistribuirea personalului.

### Limitele predicțiilor

- Modelul prezice *dacă* apar ore suplimentare, nu *câte* ore sau ce sumă; costul este estimat indirect.
- Învață din tiparele istorice: schimbările de politică, crizele (de exemplu pandemia) sau reorganizările de agenții nu sunt anticipate.
- Rândurile sunt înregistrări salariale, nu persoane unice; un angajat cu mai multe înregistrări este numărat de mai multe ori.
- Doar anul de test (`split = test`) reflectă performanța reală; pe anii de antrenare valorile sunt optimiste.
- Probabilitățile sunt utile agregat (agenție, borough); pentru decizii despre persoane individuale nu sunt potrivite.